In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [ ]:
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage: {memory_mb:.2f} MB")

missing_pct = (df.isnull().sum() / len(df)) * 100
print(f"\nMissing value percentages (%):\n{missing_pct.round(2)}")

# 3. Number of unique values per column
print(f"\nUnique values per column:\n{df.nunique()}")

# 4. Basic statistics for numeric columns (min, max, mean, median, std)
# describe() defaults to numeric columns. We transpose (.T) for readability 
# and select the specific columns required.
stats = df.describe().T[['min', 'max', 'mean', '50%', 'std']]
stats.rename(columns={'50%': 'median'}, inplace=True)
print(f"\nBasic statistics for numeric columns:\n{stats.round(2)}")


Memory usage: 126.18 MB

Missing value percentages (%):
InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

Unique values per column:
InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

Basic statistics for numeric columns:
                             min                  max  \
Quantity                -80995.0              80995.0   
InvoiceDate  2010-12-01 08:26:00  2011-12-09 12:50:00   
UnitPrice              -11062.06              38970.0   
CustomerID               12346.0              18287.0   

                                      mean               median          std  
Quantity                           9.55225                  3.0   218.081158  
InvoiceDate  2011-07-04 13:34:57.156386048  2011-07-19 17:17:00      

In [ ]:
missing_rows_mask = df.isnull().any(axis=1)

df_missing_only = df[missing_rows_mask]

df_missing_only

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,2011-12-09 10:26:00,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,2011-12-09 10:26:00,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,2011-12-09 10:26:00,10.79,NaN,United Kingdom


In [8]:
import pandas as pd

# Assuming you already loaded df
# df = pd.read_excel("Online Retail.xlsx")

# Create indicator columns
df['Missing_CustomerID'] = df['CustomerID'].isnull()
df['Missing_Description'] = df['Description'].isnull()

In [9]:
print("\n--- Investigating Missing Descriptions ---")

# 1. How does UnitPrice look when Description is missing vs present?
desc_price_stats = df.groupby('Missing_Description')['UnitPrice'].describe()
print("\nUnitPrice stats based on Missing Description:")
print(desc_price_stats[['count', 'mean', 'min', 'max']])

# 2. Are CustomerIDs also missing when Description is missing?
# A crosstab shows the frequency of combinations
desc_cust_cross = pd.crosstab(df['Missing_Description'], df['Missing_CustomerID'])
print("\nCrosstab: Missing Description vs Missing CustomerID:")
print(desc_cust_cross)


--- Investigating Missing Descriptions ---

UnitPrice stats based on Missing Description:
                        count      mean       min      max
Missing_Description                                       
False                540455.0  4.623519 -11062.06  38970.0
True                   1454.0  0.000000      0.00      0.0

Crosstab: Missing Description vs Missing CustomerID:
Missing_CustomerID    False   True 
Missing_Description                
False                406829  133626
True                      0    1454


In [10]:
print("\n--- Investigating Missing CustomerIDs ---")

# 1. Compare purchasing behavior (Quantity and UnitPrice)
cust_numeric_stats = df.groupby('Missing_CustomerID')[['Quantity', 'UnitPrice']].describe()

print("\nQuantity stats based on Missing CustomerID:")
# .xs is just a pandas trick to grab a specific sub-column easily from the grouped describe()
print(cust_numeric_stats.xs('Quantity', axis=1, level=0)[['mean', 'min', 'max']])

print("\nUnitPrice stats based on Missing CustomerID:")
print(cust_numeric_stats.xs('UnitPrice', axis=1, level=0)[['mean', 'min', 'max']])

# 2. Compare the Country distribution
print("\nTop 5 Countries when CustomerID is Missing (Percentages):")
missing_cust_countries = df[df['Missing_CustomerID']]['Country'].value_counts(normalize=True).head() * 100
print(missing_cust_countries.round(2))

print("\nTop 5 Countries when CustomerID is Present (Percentages):")
present_cust_countries = df[~df['Missing_CustomerID']]['Country'].value_counts(normalize=True).head() * 100
print(present_cust_countries.round(2))


--- Investigating Missing CustomerIDs ---

Quantity stats based on Missing CustomerID:
                         mean      min      max
Missing_CustomerID                             
False               12.061303 -80995.0  80995.0
True                 1.995573  -9600.0   5568.0

UnitPrice stats based on Missing CustomerID:
                        mean       min       max
Missing_CustomerID                              
False               3.460471      0.00  38970.00
True                8.076577 -11062.06  17836.46

Top 5 Countries when CustomerID is Missing (Percentages):
Country
United Kingdom    98.90
EIRE               0.53
Hong Kong          0.21
Unspecified        0.15
Switzerland        0.09
Name: proportion, dtype: float64

Top 5 Countries when CustomerID is Present (Percentages):
Country
United Kingdom    88.95
Germany            2.33
France             2.09
EIRE               1.84
Spain              0.62
Name: proportion, dtype: float64
